In [3]:
import pandas as pd
data = pd.read_csv('../../../data/thelook/procesado/orders_price_client.csv')
data.isnull().sum()
data.dropna(inplace=True)
data.head(5)

,order_id,user_id,created_at,sale_price,product_id
0,10826,8614,2020-06-28 00:40:17,2.5,13606
1,13243,10505,2022-03-01 05:18:44,2.5,13606
2,53140,42340,2021-04-11 01:31:52,2.5,13606
3,104681,83704,2022-03-29 21:37:06,2.5,13606
4,117931,94363,2021-02-17 09:28:17,2.5,13606


In [6]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import MiniBatchKMeans, Birch
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score

# 1. CARGA Y PREPARACIÓN (Input: tus 4 columnas)
df = pd.read_csv('../../../data/thelook/procesado/orders_price_client.csv')
df['created_at'] = pd.to_datetime(df['created_at'])
df = df.dropna(subset=['created_at'])

# 2. INGENIERÍA DE CARACTERÍSTICAS (RFM)
now = df['created_at'].max() + pd.Timedelta(days=1)
rfm = df.groupby('user_id').agg({
    'created_at': lambda x: (now - x.max()).days,
    'order_id': 'nunique',
    'sale_price': 'sum'
}).rename(columns={'created_at': 'Recency', 'order_id': 'Frequency', 'sale_price': 'Monetary'})

# Escalado (Esencial para ML)
scaler = StandardScaler()
rfm_scaled = scaler.fit_transform(rfm)

# 3. COMPETICIÓN DE ALGORITMOS
# Usamos un sample para calcular la métrica de silueta si el dataset es muy grande
sample_idx = np.random.choice(len(rfm_scaled), min(100000, len(rfm_scaled)), replace=False)
rfm_sample = rfm_scaled[sample_idx]

best_score = -1
best_model_name = ""
final_labels = None

# Definimos los modelos a probar (K=4 es un estándar útil para RFM)
k = 4
modelos = {
    'KMeans': MiniBatchKMeans(n_clusters=k, random_state=42, n_init=3),
    'BIRCH': Birch(n_clusters=k),
    'GMM': GaussianMixture(n_components=k, random_state=42)
}

print("Iniciando competición de modelos...")

for nombre, modelo in modelos.items():
    labels = modelo.fit_predict(rfm_scaled)
    score = silhouette_score(rfm_sample, labels[sample_idx])
    print(f"{nombre} - Silhouette Score: {score:.4f}")
    
    if score > best_score:
        best_score = score
        best_model_name = nombre
        final_labels = labels

print(f"\n--- Ganador: {best_model_name} ---")

# 4. ASIGNACIÓN Y ETIQUETADO AUTOMÁTICO
rfm['Cluster'] = final_labels

# Lógica para nombrar segmentos basada en el valor monetario medio
stats = rfm.groupby('Cluster')['Monetary'].mean().sort_values(ascending=False)
nombres = {stats.index[0]: 'VIP', stats.index[-1]: 'Perdidos'}
# (Rellenar el resto como 'Promedio')
rfm['Segmento'] = rfm['Cluster'].map(lambda x: nombres.get(x, 'Potencial'))

Iniciando competición de modelos...
KMeans - Silhouette Score: 0.3192
BIRCH - Silhouette Score: 0.4425
GMM - Silhouette Score: 0.2372

--- Ganador: BIRCH ---
